# Caracal Base 3B - Kaggle T4 continued pretrain

Stack: Qwen2.5-Coder-3B-Instruct + LoRA r=32 + Unsloth + TRL SFTTrainer.

**Antes de Run All:**
1. Settings -> Accelerator -> GPU T4 x2
2. Settings -> Internet -> ON
3. Settings -> Persistence -> Variables and Files
4. Editar as 5 variaveis abaixo

In [ ]:
SESSION = 1
FOUNDER_HANDLE = "pamf2"
RESUME_DATASET = None
OUTPUT_DATASET_SLUG = "caracal-base-3b-s01"
STEPS = 5000
print(f"Session {SESSION} | founder={FOUNDER_HANDLE} | steps={STEPS}")

In [ ]:
!pip install -q unsloth 'trl>=0.12.0' 'datasets>=3.0.0' 'peft>=0.13.0' 'bitsandbytes>=0.44.0' kaggle

In [ ]:
import os

if not os.path.exists("/kaggle/working/caracal-1"):
    !git clone --depth 1 -b dev https://github.com/iterate-labs-ai/caracal-1.git /kaggle/working/caracal-1
%cd /kaggle/working/caracal-1
!git rev-parse HEAD

In [ ]:
CKPT_IN = None
if RESUME_DATASET:
    !kaggle datasets download -d {RESUME_DATASET} -p /kaggle/working/ckpt-in --unzip
    CKPT_IN = "/kaggle/working/ckpt-in"
    print(f"Checkpoint pulled to {CKPT_IN}")
else:
    print("Cold start")

In [ ]:
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
OUTPUT = '/kaggle/working/ckpt-out'
cmd = (
    f'python -u train/continued_pretrain.py '
    f'--steps-to-run {STEPS} --output {OUTPUT} '
    f'--batch-size 1 --grad-accum 16 --max-seq-length 2048 '
    f'--save-every 50 --max-per-dataset 50000'
)
if CKPT_IN:
    cmd += f' --resume-from {CKPT_IN}'
print('Running:', cmd, flush=True)
!{cmd}

In [ ]:
import json
import os

if os.path.exists(f'{OUTPUT}/adapter_config.json'):
    !python eval/run_probe.py --adapter {OUTPUT} --out /kaggle/working/probe-s{SESSION}.json --max-new 64
    try:
        rep = json.loads(open(f'/kaggle/working/probe-s{SESSION}.json').read())
        print(f"mean_ppl={rep['mean_ppl']:.2f} cwe_hit_rate={rep['cwe_hit_rate']}", flush=True)
    except Exception as e:
        print(f'probe-eval failed: {e}', flush=True)
else:
    print(f'no adapter at {OUTPUT}, skipping probe eval', flush=True)

In [ ]:
import json

meta = {
    "title": f"Caracal Base 3B - Sessao {SESSION}",
    "id": f"{FOUNDER_HANDLE}/{OUTPUT_DATASET_SLUG}",
    "licenses": [{"name": "Apache-2.0"}],
    "description": f"Continued pretrain Qwen2.5-Coder-3B + LoRA. Sessao {SESSION} steps={STEPS}.",
    "keywords": ["caracal", "cybersec", "qwen", "lora", "iterate-labs"],
}
with open(f"{OUTPUT}/dataset-metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
!kaggle datasets create -p {OUTPUT} --public

## Final

1. Anotar slug do dataset criado (output acima).
2. PR atualizando SCHEDULE.md sua linha pending -> done.
3. RESUME_DATASET pra proxima sessao: `<seu_handle>/<OUTPUT_DATASET_SLUG>`.